# 06 — Synthetic Control
**Autores clave:** Abadie & Gardeazabal (2003) · Abadie, Diamond & Hainmueller (2010, 2015) · Arkhangelsky et al. (2021) *Synthetic Difference-in-Differences*

## Motivación
¿Qué pasa cuando solo hay **una unidad tratada** (un país, un estado, una empresa) y no hay un grupo control obvio?

DiD clásico requiere múltiples tratados y controles. Synthetic Control construye un control sintético como **combinación convexa de unidades donoras** que reproduce la trayectoria pre-tratamiento del tratado.

## Estimador (Abadie, Diamond & Hainmueller, 2010)
$$\hat{Y}_{1t}(0) = \sum_{j=2}^{J+1} w_j^* Y_{jt}$$

Los pesos $W^* = (w_2^*, \ldots, w_{J+1}^*)$ se eligen para minimizar la distancia entre la unidad tratada y el control sintético en el período pre-tratamiento:
$$W^* = \arg\min_{W} \| X_1 - X_0 W \|_V \quad \text{sujeto a } w_j \geq 0, \sum_j w_j = 1$$

## Inferencia: Placebo Tests
Sin la distribución asintótica estándar (solo $J+1$ unidades), la inferencia se basa en **reasignar el tratamiento a cada unidad control** y comparar el efecto estimado contra la distribución de efectos placebo.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Diseño: impacto de una ley de control de tabaco en el consumo de cigarrillos
# Inspirado en Abadie, Diamond & Hainmueller (2010) — California Tobacco Control Program
T_pre   = 15   # períodos pre-tratamiento (1970-1984)
T_post  = 10   # períodos post-tratamiento (1985-1994)
T       = T_pre + T_post
J       = 10   # unidades donoras (otros estados)
TRUE_EFFECT = -8.0  # reducción en consumo (cigarrillos per cápita)

# Factor latente común (tendencia de largo plazo)
factor = np.cumsum(np.random.normal(0, 1, T)) + np.linspace(0, 5, T)

# Cargas de cada unidad sobre el factor latente
loadings = np.abs(np.random.normal(1, 0.3, J + 1))

# Construir panel (J+1 unidades × T períodos)
Y = np.outer(loadings, factor) + np.random.normal(0, 2, (J+1, T))
Y += np.random.normal(50, 10, (J+1, 1))  # nivel base diferente por unidad

# Unidad 0 = tratada. Agregar efecto post-tratamiento
Y[0, T_pre:] += TRUE_EFFECT

periods = np.arange(-T_pre, T_post)  # tiempo relativo
unit_names = ['California'] + [f'State_{j}' for j in range(1, J+1)]

# DataFrame largo
rows = []
for j, name in enumerate(unit_names):
    for t, period in enumerate(periods):
        rows.append({'unit': name, 'unit_id': j, 'period': period,
                     'treated': int(j == 0), 'post': int(period >= 0),
                     'Y': Y[j, t]})
df = pd.DataFrame(rows)
print(f'Panel: {J+1} unidades × {T} períodos | Efecto verdadero: {TRUE_EFFECT}')

## 1 — Construir el control sintético

In [ ]:
# Matrices pre-tratamiento
Y_pre    = Y[:, :T_pre]   # todas las unidades, período pre
y1_pre   = Y_pre[0]       # unidad tratada pre
Y0_pre   = Y_pre[1:]      # donoras pre (J × T_pre)

def synthetic_control(y1_pre, Y0_pre):
    """Encuentra pesos óptimos W* minimizando ||y1 - Y0'w||² con w≥0, Σw=1."""
    J = Y0_pre.shape[0]

    def objective(w):
        synth = Y0_pre.T @ w
        return np.sum((y1_pre - synth)**2)

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0, 1)] * J
    w0 = np.ones(J) / J
    res = minimize(objective, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x

weights = synthetic_control(y1_pre, Y0_pre)

# Serie del control sintético (todos los períodos)
synth_series = Y[1:].T @ weights  # T × 1

print('Pesos del control sintético:')
weight_df = pd.DataFrame({'Unidad': unit_names[1:], 'Peso': weights.round(4)})
print(weight_df[weight_df['Peso'] > 0.01].sort_values('Peso', ascending=False).to_string(index=False))
print(f'\nPre-RMSPE (fit pre-tratamiento): {np.sqrt(np.mean((y1_pre - synth_series[:T_pre])**2)):.4f}')

## 2 — Gráfica principal del Synthetic Control

In [ ]:
treated_series = Y[0]  # California observada
gap_series     = treated_series - synth_series

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: tratada vs sintético
ax = axes[0]
ax.plot(periods, treated_series, 'o-', color='#f72585', linewidth=2.5, markersize=5,
        label='California (tratada)')
ax.plot(periods, synth_series, 's--', color='#4361ee', linewidth=2.5, markersize=5,
        label='Control sintético')
ax.axvline(-0.5, color='black', linewidth=1.5, linestyle=':', label='Tratamiento')
ax.fill_between(periods[T_pre:], treated_series[T_pre:], synth_series[T_pre:],
                alpha=0.2, color='#f72585', label='Efecto estimado')
ax.set_xlabel('Período (relativo al tratamiento)')
ax.set_ylabel('Consumo cigarrillos per cápita')
ax.set_title('Synthetic Control — Abadie, Diamond & Hainmueller (2010)')
ax.legend(fontsize=8)

# Panel B: gap (efecto estimado)
ax2 = axes[1]
ax2.plot(periods, gap_series, 'o-', color='#4361ee', linewidth=2, markersize=5)
ax2.axhline(0, color='black', linewidth=1)
ax2.axvline(-0.5, color='black', linewidth=1.5, linestyle=':')
ax2.axhline(TRUE_EFFECT, color='#06d6a0', linewidth=1.5, linestyle='--',
            label=f'Efecto verdadero = {TRUE_EFFECT}')
ax2.fill_between(periods[:T_pre], gap_series[:T_pre], 0, alpha=0.1, color='#4361ee')
ax2.fill_between(periods[T_pre:], gap_series[T_pre:], 0, alpha=0.2, color='#f72585')
ax2.set_xlabel('Período (relativo al tratamiento)')
ax2.set_ylabel('Gap: California − Sintético')
ax2.set_title(f'Gap estimado\nATT post: {gap_series[T_pre:].mean():.2f}  (verdadero: {TRUE_EFFECT})')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 3 — Inferencia: In-Time y In-Space Placebo Tests

In [ ]:
# In-space placebo: aplicar synthetic control a cada donora como si fuera el tratado
placebo_gaps = []

for j in range(1, J + 1):
    # Donoras = todas excepto j y la unidad tratada original
    donor_ids   = [k for k in range(J+1) if k != 0 and k != j]
    y_placebo   = Y[j, :T_pre]
    Y0_placebo  = Y[donor_ids, :T_pre]

    try:
        w_p   = synthetic_control(y_placebo, Y0_placebo)
        synth_p = Y[donor_ids].T @ w_p
        # Excluir placebos con mal fit pre-tratamiento (ratio > 5× el tratado)
        pre_rmspe_p = np.sqrt(np.mean((y_placebo - synth_p[:T_pre])**2))
        pre_rmspe_t = np.sqrt(np.mean((y1_pre - synth_series[:T_pre])**2))
        if pre_rmspe_p < 5 * pre_rmspe_t:
            placebo_gaps.append(Y[j] - synth_p)
    except Exception:
        pass

print(f'Placebos calculados: {len(placebo_gaps)} (excluidos por mal pre-fit: {J - len(placebo_gaps)})')

# p-value: proporción de placebos con efecto post >= efecto del tratado
effect_treated = gap_series[T_pre:].mean()
effect_placebos = [g[T_pre:].mean() for g in placebo_gaps]
p_val = np.mean(np.abs(effect_placebos) >= np.abs(effect_treated))

print(f'Efecto post-tratamiento: {effect_treated:.3f}')
print(f'p-value placebo:         {p_val:.4f}  (prob de obtener efecto ≥ observado por azar)')

fig, ax = plt.subplots(figsize=(11, 5))
for gap in placebo_gaps:
    ax.plot(periods, gap, color='#adb5bd', linewidth=1, alpha=0.5)
ax.plot(periods, gap_series, color='#f72585', linewidth=3, label='California (tratada)', zorder=5)
ax.axhline(0, color='black', linewidth=1)
ax.axvline(-0.5, color='black', linewidth=1.5, linestyle=':', label='Tratamiento')
ax.set_xlabel('Período')
ax.set_ylabel('Gap')
ax.set_title(f'Placebo Test In-Space — Abadie, Diamond & Hainmueller (2010)\np-value = {p_val:.3f}')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Resumen

| Paso | Qué hacer | Por qué |
|---|---|---|
| 1. Pre-fit | Pre-RMSPE → 0 | Valida que el sintético replica al tratado |
| 2. Pesos | $w_j \geq 0$, $\sum w_j = 1$ | Convexidad → no extrapolación |
| 3. Gap post | $\hat{Y}_{1t} - \hat{Y}_{1t}(0)$ | Efecto causal estimado |
| 4. Placebos in-space | Mismo procedimiento a donoras | Distribución nula para inferencia |
| 5. Placebos in-time | Tratamiento ficticio en pre-período | Cheque de especificación |

**Referencias:**
- Abadie, A. & Gardeazabal, J. (2003). The economic costs of conflict. *American Economic Review*, 93(1).
- Abadie, A., Diamond, A. & Hainmueller, J. (2010). Synthetic control methods for comparative case studies. *JASA*, 105(490).
- Abadie, A., Diamond, A. & Hainmueller, J. (2015). Comparative politics and the synthetic control method. *American Journal of Political Science*, 59(2).
- Arkhangelsky, D., Athey, S., Hirshberg, D.A., Imbens, G.W. & Wager, S. (2021). Synthetic difference-in-differences. *American Economic Review*, 111(12).

**Siguiente:** `07_limited_dependent_variables.ipynb`